# 30 — Supplier Risk Scorer

Pass in a supplier list with countries and get a geopolitical risk register backed by World Bank governance data — no API key, no subscription. The LLM converts raw governance scores into ranked risk tiers with specific mitigation actions.

In [ ]:
!pip install -q openai python-dotenv pydantic

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = 'sk-...'  # replace

In [ ]:
from pydantic import BaseModel, Field

class GovernanceIndicators(BaseModel):
    political_stability: float | None = Field(description='WGI Political Stability score (-2.5 to 2.5).')
    rule_of_law: float | None = Field(description='WGI Rule of Law score (-2.5 to 2.5).')
    control_of_corruption: float | None = Field(description='WGI Control of Corruption score (-2.5 to 2.5).')
    regulatory_quality: float | None = Field(description='WGI Regulatory Quality score (-2.5 to 2.5).')
    data_year: int | None = Field(description='Year of the governance data.')

class SupplierRisk(BaseModel):
    supplier: str = Field(description='Supplier name.')
    country: str = Field(description='Country of operations.')
    country_code: str = Field(description='ISO alpha-3 code.')
    governance_indicators: GovernanceIndicators = Field(description='Raw WGI scores.')
    geopolitical_risk_score: int = Field(description='Composite risk 0-100 (higher = more risk).')
    risk_tier: str = Field(description='LOW, MEDIUM, HIGH, or CRITICAL.')
    key_risks: list[str] = Field(description='Up to 3 governance risks driving the score.')
    mitigation: str = Field(description='One concrete mitigation action.')

class SupplierRiskRegister(BaseModel):
    suppliers_assessed: int = Field(description='Total suppliers scored.')
    critical_count: int = Field(description='Suppliers rated CRITICAL.')
    high_count: int = Field(description='Suppliers rated HIGH.')
    suppliers: list[SupplierRisk] = Field(description='All assessments, sorted by risk score descending.')
    portfolio_summary: str = Field(description='2-3 sentence portfolio risk assessment and priority action.')

In [ ]:
import json
import urllib.request
from openai import OpenAI

WB_URL = 'https://api.worldbank.org/v2/country/{code}/indicator/{indicator}?format=json&mrv=1&per_page=1'

WGI = {
    'political_stability': 'PV.EST',
    'rule_of_law': 'RL.EST',
    'control_of_corruption': 'CC.EST',
    'regulatory_quality': 'RQ.EST',
}

COUNTRY_MAP = {
    'china': 'CHN', 'usa': 'USA', 'united states': 'USA', 'germany': 'DEU',
    'india': 'IND', 'mexico': 'MEX', 'brazil': 'BRA', 'vietnam': 'VNM',
    'taiwan': 'TWN', 'south korea': 'KOR', 'japan': 'JPN', 'malaysia': 'MYS',
    'indonesia': 'IDN', 'thailand': 'THA', 'bangladesh': 'BGD', 'pakistan': 'PAK',
    'myanmar': 'MMR', 'cambodia': 'KHM', 'nigeria': 'NGA', 'ukraine': 'UKR',
    'russia': 'RUS', 'philippines': 'PHL', 'sri lanka': 'LKA',
}

SYSTEM = (
    'You are a supply chain risk analyst. Given World Bank WGI scores (-2.5 to +2.5, higher is better):\n'
    '- Convert scores to geopolitical_risk_score 0-100 (lower WGI = higher risk)\n'
    '- Assign risk_tier: LOW (0-25), MEDIUM (26-50), HIGH (51-75), CRITICAL (76-100)\n'
    '- List up to 3 key governance risks driving the score\n'
    '- Suggest one concrete mitigation action matching the risk tier\n'
    '- Sort suppliers by risk score descending\n'
    '- Write a 2-3 sentence portfolio_summary\n'
    'Use only provided WGI data.'
)

def _get(url):
    req = urllib.request.Request(url, headers={'User-Agent': 'agent-use-cases'})
    with urllib.request.urlopen(req, timeout=10) as r:
        return json.loads(r.read())

def _wgi(code):
    scores = {k: None for k in WGI}
    year = None
    for field, indicator in WGI.items():
        try:
            data = _get(WB_URL.format(code=code.lower(), indicator=indicator))
            if len(data) >= 2 and data[1]:
                val = data[1][0].get('value')
                if val is not None:
                    scores[field] = round(float(val), 3)
                if not year and data[1][0].get('date'):
                    year = int(data[1][0]['date'])
        except Exception:
            pass
    return {**scores, 'data_year': year}

def score(suppliers):
    data = []
    for name, country in suppliers:
        code = COUNTRY_MAP.get(country.lower().strip(), country.upper()[:3])
        data.append({'supplier': name, 'country': country, 'country_code': code, 'governance_indicators': _wgi(code)})
    client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])
    r = client.beta.chat.completions.parse(
        model='gpt-4o-mini',
        messages=[{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': f'Suppliers: {len(suppliers)}\n\n{json.dumps(data, indent=2)}'}],
        response_format=SupplierRiskRegister
    )
    reg = r.choices[0].message.parsed
    reg.suppliers_assessed = len(suppliers)
    return reg

In [ ]:
# Apparel manufacturer -- Southeast Asia / South Asia supply chain
APPAREL = [
    ('Apex Textiles', 'Bangladesh'),
    ('Golden Thread Co.', 'Vietnam'),
    ('Sunrise Fabrics', 'Myanmar'),
    ('Pacific Yarn Ltd.', 'Taiwan'),
    ('Continental Mills', 'Germany'),
]

reg = score(APPAREL)
print(f'Assessed: {reg.suppliers_assessed} | Critical: {reg.critical_count} | High: {reg.high_count}')
for s in reg.suppliers:
    gi = s.governance_indicators
    print(f'\n[{s.risk_tier}] {s.supplier} — {s.country} | Score: {s.geopolitical_risk_score}/100')
    if gi.political_stability is not None:
        print(f'  WGI: stability={gi.political_stability:+.2f}  law={gi.rule_of_law:+.2f}  corruption={gi.control_of_corruption:+.2f}')
    print(f'  Risks: {" | ".join(s.key_risks)}')
    print(f'  Mitigation: {s.mitigation}')
print(f'\nPortfolio: {reg.portfolio_summary}')

In [ ]:
# Electronics manufacturer -- Asia + Africa mix
ELECTRONICS = [
    ('Shenzhen Components', 'China'),
    ('Chennai Circuit Board', 'India'),
    ('Kyoto Precision Parts', 'Japan'),
    ('Manila Electronics', 'Philippines'),
    ('Lagos Assembly Works', 'Nigeria'),
]

reg2 = score(ELECTRONICS)
print(f'Assessed: {reg2.suppliers_assessed} | Critical: {reg2.critical_count} | High: {reg2.high_count}')
for s in reg2.suppliers:
    print(f'\n[{s.risk_tier}] {s.supplier} — {s.country} | Score: {s.geopolitical_risk_score}/100')
    print(f'  Mitigation: {s.mitigation}')
print(f'\nPortfolio: {reg2.portfolio_summary}')

## Exercise

Add a supplier from Russia, Ukraine, or Pakistan and re-run. Compare how `portfolio_summary` changes when a CRITICAL-tier supplier enters the mix. The World Bank data is live — scores reflect the most recent annual update.